# Modelo 1: Expected Goals (xG) - Predicción de Probabilidad de Gol
**Taller 2 - Machine Learning I**

Este notebook contiene el flujo completo para el desarrollo del modelo de **Expected Goals**. Utiliza datos de eventos espaciales para entrenar un clasificador logístico capaz de discernir la probabilidad de que un tiro termine en gol.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style="whitegrid")

## 1. Carga de Datos y Exploración de Tiros
Cargamos los datos crudos y filtramos exclusivamente los eventos de tipo **tiro** (`is_shot == 1`).

In [ ]:
events_df = pd.read_csv('../../data/events.csv')
shots_df = events_df[events_df['is_shot'] == 1].copy()
print(f"Total de tiros registrados: {len(shots_df)}")
print(f"Tasa de conversión base (Goles reales): {(shots_df['is_goal'].mean()*100):.2f}%")

## 2. Feature Engineering: Geometría y Contexto
Calculamos variables espaciales críticas e inyectamos metadatos de los `qualifiers`.

In [ ]:
# Geometría del Arco
shots_df['dist_al_arco'] = np.sqrt((100 - shots_df['x'])**2 + (50 - shots_df['y'])**2)
shots_df['angulo_tiro'] = np.arctan2(50 - shots_df['y'], 100 - shots_df['x']).abs()

# Parseo de Cualificadores
shots_df['is_BigChance'] = shots_df['qualifiers'].str.contains('BigChance', na=False).astype(int)
shots_df['is_Penalty'] = shots_df['qualifiers'].str.contains('"Penalty"', na=False).astype(int)
shots_df['is_OneOnOne'] = shots_df['qualifiers'].str.contains('OneOnOne', na=False).astype(int)

# Cruce con Amenaza del Jugador (FPL Threat)
players_df = pd.read_csv('../../data/players.csv')
players_df['full_name'] = players_df['first_name'] + ' ' + players_df['second_name']
df_final = pd.merge(shots_df, players_df[['full_name', 'threat']], left_on='player_name', right_on='full_name', how='left')
df_final['threat'] = df_final['threat'].fillna(df_final['threat'].median())

# Guardar matriz de entrenamiento
features = ['dist_al_arco', 'angulo_tiro', 'is_BigChance', 'is_Penalty', 'is_OneOnOne', 'threat']
df_final[features + ['is_goal', 'player_name']].to_csv('../../data/xg_train.csv', index=False)
print("Matriz ../../data/xg_train.csv guardada.")

## 3. Validación de Supuestos (Multicolinealidad - VIF)
Aseguramos que nuestras variables sean independientes (VIF < 5).

In [ ]:
X_vif = df_final[features].copy()
X_vif['intercept'] = 1
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]
vif_data.round(2)

## 4. Entrenamiento y Evaluación (Logistic Regression)
Entrenamos con penalización balanceada de clases.

In [ ]:
X = df_final[features]
y = df_final['is_goal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluación
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"ROC-AUC Score: {auc:.4f}")
print(classification_report(y_test, y_pred))

## 5. Visualización Final
Generamos la Curva ROC y la Matriz de Confusión.

In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic xG (AUC = {auc:.3f})', color='darkred')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.title('Curva ROC - Expected Goals (Modelo 1)')
plt.legend()
plt.savefig('../../img/roc_curve_xg.png')
plt.show()

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Gol', 'Gol'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión (Clase Balanceada)')
plt.savefig('../../img/confusion_matrix_xg.png')
plt.show()